# Actor & Director Credibility from Public Movie Data

This notebook computes **credibility metrics** for actors and directors using a public movie dataset (The Movies Dataset, based on TMDb).[web:87]

We aggregate historical box office and rating performance per person and export two CSVs:

- `actor_credibility_from_public.csv`
- `director_credibility_from_public.csv`

These can be joined into our IMDb project to provide features such as **cast star power** and **director track record** for movie‑level prediction tasks.


In [ ]:
import pandas as pd
import numpy as np
import ast
import os

# Base path where movies_metadata.csv and credits.csv live
BASE_PATH = "data/archive"

MOVIES_FILE  = os.path.join(BASE_PATH, "movies_metadata.csv")
CREDITS_FILE = os.path.join(BASE_PATH, "credits.csv")

ACTOR_OUT    = os.path.join(BASE_PATH, "actor_credibility_from_public.csv")
DIRECTOR_OUT = os.path.join(BASE_PATH, "director_credibility_from_public.csv")

CUTOFF_YEAR = 2020          # use movies released before this year
RATING_HIT_THRESHOLD = 7.0  # defines a "hit" movie


## 1. Load and prepare movie data

We use `movies_metadata.csv` from The Movies Dataset.[web:87]

From this file we extract:

- `movie_id` (numeric TMDb ID)
- `revenue` (worldwide box office)
- `vote_average`, `vote_count` (rating and popularity)
- `release_year` (derived from `release_date`)

We keep only movies with valid revenue and ratings, then restrict to a **historical window** (`release_year < 2020`) so that actor/director credibility is based on **past performance** only. We also define a binary `is_hit` flag for movies with `vote_average ≥ 7.0`.


In [ ]:
# Load movies_metadata.csv
movies = pd.read_csv(MOVIES_FILE, low_memory=False)

movies = movies[["id", "revenue", "vote_average", "vote_count", "release_date"]].copy()
movies = movies.rename(columns={"id": "movie_id"})

# Clean numeric fields
movies["movie_id"] = pd.to_numeric(movies["movie_id"], errors="coerce")
movies["revenue"] = pd.to_numeric(movies["revenue"], errors="coerce")
movies["vote_average"] = pd.to_numeric(movies["vote_average"], errors="coerce")
movies["vote_count"] = pd.to_numeric(movies["vote_count"], errors="coerce")

# Drop rows without key info
movies = movies.dropna(subset=["movie_id", "revenue", "vote_average", "vote_count"])

# Extract release_year
movies["release_year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year
movies = movies.dropna(subset=["release_year"])
movies["release_year"] = movies["release_year"].astype(int)

# Historical subset
movies_hist = movies[movies["release_year"] < CUTOFF_YEAR].copy()

# Define hit flag
movies_hist["is_hit"] = (movies_hist["vote_average"] >= RATING_HIT_THRESHOLD).astype(int)

movies_hist.head()


## 2. Load and parse credits (cast & crew)

We use `credits.csv`, which contains JSON‑encoded lists for each movie:

- `cast`: actors with `id` and `name`
- `crew`: crew members with `id`, `name`, `job`, etc.[web:87]

From this we build:

- an **actor–movie table**: one row per `(movie_id, actor_id, actor_name)`
- a **director–movie table**: one row per `(movie_id, director_id, director_name)` where `job == "Director"`


In [ ]:
credits = pd.read_csv(CREDITS_FILE, low_memory=False)

def parse_json_list(cell):
    try:
        return ast.literal_eval(cell)
    except Exception:
        return []

credits["cast_parsed"] = credits["cast"].apply(parse_json_list)
credits["crew_parsed"] = credits["crew"].apply(parse_json_list)

# Build actor–movie table
actor_rows = []
for _, row in credits.iterrows():
    mid = row["id"]
    for person in row["cast_parsed"]:
        pid = person.get("id")
        name = person.get("name")
        if pid is None:
            continue
        actor_rows.append((mid, pid, name))

actors_df = pd.DataFrame(actor_rows, columns=["movie_id", "person_id", "person_name"])

# Build director–movie table
director_rows = []
for _, row in credits.iterrows():
    mid = row["id"]
    for person in row["crew_parsed"]:
        if person.get("job") == "Director":
            pid = person.get("id")
            name = person.get("name")
            if pid is None:
                continue
            director_rows.append((mid, pid, name))

directors_df = pd.DataFrame(director_rows, columns=["movie_id", "person_id", "person_name"])

actors_df.head(), directors_df.head()


## 3. Attach box office and rating to each credit

We now join the actor/director tables with the historical movie subset to attach:

- `revenue`
- `vote_average`
- `vote_count`
- `is_hit`

to every actor‑movie and director‑movie row.


In [ ]:
actor_movie = actors_df.merge(
    movies_hist[["movie_id", "revenue", "vote_average", "vote_count", "is_hit"]],
    on="movie_id",
    how="inner",
)

director_movie = directors_df.merge(
    movies_hist[["movie_id", "revenue", "vote_average", "vote_count", "is_hit"]],
    on="movie_id",
    how="inner",
)

len(actor_movie), len(director_movie)


## 4. Actor credibility metrics

For each actor (`person_id`, `person_name`), we aggregate over all their historical movies:

- `actor_avg_box_office`: mean of `revenue`
- `actor_median_box_office`: median `revenue`
- `actor_max_box_office`: maximum `revenue`
- `actor_movie_count`: number of movies (experience)
- `actor_hit_rate`: fraction of movies where `is_hit == 1` (rating ≥ 7.0)

We also compute a composite **`actor_credibility_score`** combining box office power, hit rate, and experience:

\[
\text{actor\_credibility\_score} =
0.5 \cdot \log(1 + \text{avg\_box\_office}) +
0.3 \cdot (10 \cdot \text{hit\_rate}) +
0.2 \cdot \log(1 + \text{movie\_count})
\]

Actors with very few movies (e.g., `< 3`) are filtered out to reduce noise.


In [ ]:
def weighted_mean(x, w):
    s = (x * w).sum()
    return s / w.sum() if w.sum() > 0 else np.nan

g_actor = actor_movie.groupby(["person_id", "person_name"])

actor_stats = g_actor.apply(
    lambda g: pd.Series({
        "actor_avg_box_office"    : g["revenue"].mean(),
        "actor_median_box_office" : g["revenue"].median(),
        "actor_max_box_office"    : g["revenue"].max(),
        "actor_movie_count"       : len(g),
        "actor_hit_rate"          : g["is_hit"].mean(),
    })
).reset_index()

# Composite credibility score
actor_stats["actor_credibility_score"] = (
    0.5 * np.log1p(actor_stats["actor_avg_box_office"]) +
    0.3 * (actor_stats["actor_hit_rate"] * 10) +
    0.2 * np.log1p(actor_stats["actor_movie_count"])
)

# Filter actors with minimal history
MIN_MOVIES = 3
actor_stats = actor_stats[actor_stats["actor_movie_count"] >= MIN_MOVIES].copy()

actor_stats.head()


## 5. Director credibility metrics

For each director we compute analogous metrics:

- `director_avg_box_office`
- `director_max_box_office`
- `director_movie_count`
- `director_hit_rate`

and a composite **`director_credibility_score`**:

\[
\text{director\_credibility\_score} =
0.5 \cdot \log(1 + \text{avg\_box\_office}) +
0.3 \cdot (10 \cdot \text{hit\_rate}) +
0.2 \cdot \log(1 + \text{movie\_count})
\]

Directors with very few movies are also filtered out.


In [ ]:
g_dir = director_movie.groupby(["person_id", "person_name"])

director_stats = g_dir.apply(
    lambda g: pd.Series({
        "director_avg_box_office" : g["revenue"].mean(),
        "director_max_box_office" : g["revenue"].max(),
        "director_movie_count"    : len(g),
        "director_hit_rate"       : g["is_hit"].mean(),
    })
).reset_index()

director_stats["director_credibility_score"] = (
    0.5 * np.log1p(director_stats["director_avg_box_office"]) +
    0.3 * (director_stats["director_hit_rate"] * 10) +
    0.2 * np.log1p(director_stats["director_movie_count"])
)

director_stats = director_stats[director_stats["director_movie_count"] >= MIN_MOVIES].copy()

director_stats.head()


## 6. Save credibility datasets

Finally, we export the aggregated tables as CSV files:

- `actor_credibility_from_public.csv`
- `director_credibility_from_public.csv`

These files can be merged into our IMDb project by matching actors/directors via TMDb IDs or names to create features like **cast_mean_credibility** and **director_credibility_score** in movie‑level models.


In [ ]:
actor_stats.to_csv(ACTOR_OUT, index=False)
director_stats.to_csv(DIRECTOR_OUT, index=False)

print(f"Saved actor credibility to   {ACTOR_OUT}")
print(f"Saved director credibility to {DIRECTOR_OUT}")